# GINO Particle U/gradU Analysis

Analysis notebook matched to `GINO_1to2.ipynb` and the saved artifacts in `FINAL/result/task1`. It assumes the processed particle dataset is available at `FINAL/processed_data/particle_ugradu_dataset.npz` unless overridden below.


In [ ]:
from __future__ import annotations

import inspect
import json
import math
import os
import random
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from neuralop.models import GINO
    GINO_IMPORT_ERROR = None
except Exception as error:
    GINO = None
    GINO_IMPORT_ERROR = error

try:
    from neuralop.utils import count_model_params
except Exception:
    def count_model_params(model: nn.Module) -> int:
        return sum(p.numel() for p in model.parameters())

print('Torch       :', torch.__version__)
print('CUDA build  :', torch.version.cuda)
print('CUDA usable :', torch.cuda.is_available())


In [ ]:
SEED = int(os.environ.get('GINO_SEED', '42'))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def _repo_paths() -> Tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    if (cwd / 'FINAL').is_dir():
        return cwd, cwd / 'FINAL'
    if cwd.name == 'FINAL':
        return cwd.parent, cwd
    if cwd.parent.name == 'FINAL':
        return cwd.parent.parent, cwd.parent
    return cwd, cwd / 'FINAL'


def _safe_np_load(path: Path):
    try:
        return np.load(path, allow_pickle=True)
    except ModuleNotFoundError as error:
        if 'numpy._core' not in str(error):
            raise
        import numpy.core as numpy_core
        sys.modules.setdefault('numpy._core', numpy_core)
        sys.modules.setdefault('numpy._core.multiarray', np.core.multiarray)
        sys.modules.setdefault('numpy._core.numeric', np.core.numeric)
        return np.load(path, allow_pickle=True)


def _torch_load(path: Path, device: torch.device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


REPO_ROOT, FINAL_DIR = _repo_paths()
RESULTS_DIR = FINAL_DIR / 'result' / 'task1'
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path(os.environ.get('FINAL2_TASK1_UGRADU_DATASET', '')).expanduser()
if str(DATASET_PATH) == '.':
    DATASET_PATH = FINAL_DIR / 'processed_data' / 'particle_ugradu_dataset.npz'
if not DATASET_PATH.is_absolute():
    DATASET_PATH = (Path.cwd() / DATASET_PATH).resolve()
if not DATASET_PATH.exists():
    DATASET_PATH = FINAL_DIR / 'processed_data' / 'particle_ugradu_dataset.npz'

CHECKPOINT_PATH = Path(os.environ.get('GINO_ANALYSIS_CHECKPOINT', '')).expanduser()
if str(CHECKPOINT_PATH) == '.':
    CHECKPOINT_PATH = RESULTS_DIR / 'task1_particle_ugradu_gino_pointwise_best_model.pt'
if not CHECKPOINT_PATH.is_absolute():
    CHECKPOINT_PATH = (Path.cwd() / CHECKPOINT_PATH).resolve()
if not CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH = RESULTS_DIR / 'task1_particle_ugradu_gino_pointwise_best_model.pt'

HISTORY_PATH = RESULTS_DIR / 'task1_particle_ugradu_gino_pointwise_history.json'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Repo root      :', REPO_ROOT)
print('FINAL dir      :', FINAL_DIR)
print('Dataset path   :', DATASET_PATH)
print('Checkpoint     :', CHECKPOINT_PATH)
print('Plots dir      :', PLOTS_DIR)
print('Device         :', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU            :', torch.cuda.get_device_name(0))

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Missing processed dataset: {DATASET_PATH}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing checkpoint: {CHECKPOINT_PATH}')


In [ ]:
checkpoint = _torch_load(CHECKPOINT_PATH, DEVICE)
CFG = dict(checkpoint.get('config', checkpoint.get('model_config', {})))
if not CFG:
    raise RuntimeError('Checkpoint does not contain config/model_config; cannot rebuild the model safely.')

feature_names = [str(x) for x in checkpoint.get('feature_names', [])]
target_names = [str(x) for x in checkpoint.get('target_names', [])]
feature_names_all_from_checkpoint = [str(x) for x in checkpoint.get('feature_names_all', [])]

if not feature_names:
    raise RuntimeError('Checkpoint does not contain feature_names.')
if not target_names:
    raise RuntimeError('Checkpoint does not contain target_names.')

state_dict = checkpoint['model_state_dict']
state_keys = list(state_dict.keys())
architecture = 'neuralop_gino' if any(key.startswith('gno_in.') for key in state_keys) else 'pointwise_latent_gino'

print('Checkpoint tag     :', checkpoint.get('checkpoint_tag', 'unknown'))
print('Saved at UTC       :', checkpoint.get('saved_at_utc', 'unknown'))
print('Architecture       :', architecture)
print('Checkpoint config  :', json.dumps(CFG, indent=2, default=str))
print('Input features     :', feature_names)
print('Target names       :', target_names)
print('Best score         :', checkpoint.get('best_score', 'unknown'))
print('Stored metrics     :', json.dumps(checkpoint.get('metrics', {}), indent=2, default=str))


In [ ]:
dataset_file = _safe_np_load(DATASET_PATH)
feature_names_all = [str(x) for x in dataset_file['feature_names'].tolist()]
target_names_all = [str(x) for x in dataset_file['target_names'].tolist()]
frame_contexts = list(dataset_file['frame_contexts']) if 'frame_contexts' in dataset_file.files else [{} for _ in dataset_file['inputs_by_frame']]
frame_ranges = list(dataset_file['frame_ranges']) if 'frame_ranges' in dataset_file.files else [(None, i) for i in range(len(frame_contexts))]

missing_features = [name for name in feature_names if name not in feature_names_all]
missing_targets = [name for name in target_names if name not in target_names_all]
if missing_features:
    raise RuntimeError(f'Checkpoint expects features missing from dataset: {missing_features}')
if missing_targets:
    raise RuntimeError(f'Checkpoint expects targets missing from dataset: {missing_targets}')

active_input_feature_indices = [feature_names_all.index(name) for name in feature_names]
target_indices = [target_names_all.index(name) for name in target_names]
coord_feature_indices = np.asarray([feature_names_all.index(k) for k in ('x', 'y', 'z')], dtype=np.int64)

inputs_by_frame = list(dataset_file['inputs_by_frame'])
targets_by_frame = list(dataset_file['targets_by_frame'])
inputs_by_frame_raw_active = [np.asarray(x, dtype=np.float32)[:, active_input_feature_indices] for x in inputs_by_frame]
targets_by_frame_raw_active = [np.asarray(y, dtype=np.float32)[:, target_indices] for y in targets_by_frame]

input_mean_all = np.asarray(dataset_file['in_mean'], dtype=np.float32).reshape(-1)
input_std_all = np.maximum(np.asarray(dataset_file['in_std'], dtype=np.float32).reshape(-1), 1e-8)
target_mean_all = np.asarray(dataset_file['out_mean'], dtype=np.float32).reshape(-1)
target_std_all = np.maximum(np.asarray(dataset_file['out_std'], dtype=np.float32).reshape(-1), 1e-8)

input_mean = np.asarray(checkpoint.get('input_mean', input_mean_all[active_input_feature_indices]), dtype=np.float32).reshape(-1)
input_std = np.maximum(np.asarray(checkpoint.get('input_std', input_std_all[active_input_feature_indices]), dtype=np.float32).reshape(-1), 1e-8)
target_mean = np.asarray(checkpoint.get('target_mean', target_mean_all[target_indices]), dtype=np.float32).reshape(-1)
target_std = np.maximum(np.asarray(checkpoint.get('target_std', target_std_all[target_indices]), dtype=np.float32).reshape(-1), 1e-8)

if 'coord_min' in checkpoint and 'coord_span' in checkpoint:
    coord_min = np.asarray(checkpoint['coord_min'], dtype=np.float32).reshape(3)
    coord_span = np.asarray(checkpoint['coord_span'], dtype=np.float32).reshape(3)
elif 'coord_min' in dataset_file.files and 'coord_span' in dataset_file.files:
    coord_min = np.asarray(dataset_file['coord_min'], dtype=np.float32).reshape(3)
    coord_span = np.asarray(dataset_file['coord_span'], dtype=np.float32).reshape(3)
else:
    xyz_flat = np.concatenate([np.asarray(x, dtype=np.float32)[:, coord_feature_indices] for x in inputs_by_frame], axis=0)
    coord_min = np.min(xyz_flat, axis=0).astype(np.float32)
    coord_span = np.ptp(xyz_flat, axis=0).astype(np.float32)
coord_span = np.maximum(coord_span, 1e-8)

target_mean_t = torch.tensor(target_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
target_std_t = torch.tensor(target_std, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
target_var_t = torch.tensor(target_std ** 2, dtype=torch.float32, device=DEVICE).view(1, 1, -1).clamp_min(1e-12)


def as_context(obj) -> Dict:
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, 'item'):
        item = obj.item()
        if isinstance(item, dict):
            return item
    try:
        return dict(obj)
    except Exception:
        return {}


def frame_context_as_dict(frame_id: int) -> Dict:
    return as_context(frame_contexts[int(frame_id)])


def split_ids_from_checkpoint_or_file(checkpoint_key: str, file_keys: Sequence[str]) -> np.ndarray:
    if checkpoint_key in checkpoint:
        return np.asarray(checkpoint[checkpoint_key], dtype=np.int64)
    for key in file_keys:
        if key in dataset_file.files:
            return np.asarray(dataset_file[key], dtype=np.int64)
    return np.zeros((0,), dtype=np.int64)

train_frame_ids = split_ids_from_checkpoint_or_file('train_frame_ids', ['train_frame_ids'])
val_id_frame_ids = split_ids_from_checkpoint_or_file('val_id_frame_ids', ['val_id_frame_ids'])
val_angle_frame_ids = split_ids_from_checkpoint_or_file('val_angle_frame_ids', ['validation_angle_frame_ids', 'val_angle_frame_ids', 'val_frame_ids'])
test_normal_frame_ids = split_ids_from_checkpoint_or_file('test_normal_frame_ids', ['test_normal_frame_ids', 'test_frame_ids'])
test_spatial_sr_frame_ids = split_ids_from_checkpoint_or_file('test_spatial_sr_frame_ids', ['test_super_resolution_frame_ids', 'test_spatial_sr_frame_ids'])
test_temporal_sr_frame_ids = split_ids_from_checkpoint_or_file('test_temporal_sr_frame_ids', ['test_temporal_sr_frame_ids'])
test_unseen_angle_frame_ids = split_ids_from_checkpoint_or_file('test_unseen_angle_frame_ids', ['test_unseen_angle_frame_ids'])
testing_frame_ids = test_normal_frame_ids if len(test_normal_frame_ids) else val_angle_frame_ids

print('Dataset frames    :', len(inputs_by_frame))
print('Feature count     :', len(feature_names))
print('Target count      :', len(target_names))
print('Splits            :', {k: int(len(v)) for k, v in {
    'train': train_frame_ids,
    'val_id': val_id_frame_ids,
    'val_angle': val_angle_frame_ids,
    'test_normal': test_normal_frame_ids,
    'test_spatial_sr': test_spatial_sr_frame_ids,
    'test_temporal_sr': test_temporal_sr_frame_ids,
    'test_unseen_angle': test_unseen_angle_frame_ids,
}.items()})


In [ ]:
def normalize_xyz(xyz: np.ndarray) -> np.ndarray:
    return np.clip((xyz.astype(np.float32) - coord_min[None, :]) / coord_span[None, :], 0.0, 1.0).astype(np.float32)


def sample_indices(n: int, cap: Optional[int], seed: int) -> np.ndarray:
    if cap is None or cap <= 0 or n <= int(cap):
        return np.arange(n, dtype=np.int64)
    rng = np.random.default_rng(int(seed))
    return np.sort(rng.choice(n, size=int(cap), replace=False)).astype(np.int64)


def sample_frame_ids(frame_ids: Sequence[int], maximum_frames: Optional[int] = None, seed_offset: int = 0, sort_after_sampling: bool = False) -> np.ndarray:
    frame_ids = np.asarray(frame_ids, dtype=np.int64)
    if maximum_frames is None or len(frame_ids) <= int(maximum_frames):
        chosen = frame_ids.copy()
    else:
        rng = np.random.default_rng(SEED + int(seed_offset))
        chosen = rng.choice(frame_ids, size=int(maximum_frames), replace=False)
    if sort_after_sampling:
        chosen = np.asarray(sorted(chosen, key=frame_number_for_sort), dtype=np.int64)
    return chosen


def sample_dataset_indices(dataset, maximum_items: Optional[int] = None, seed_offset: int = 0) -> np.ndarray:
    indices = np.arange(len(dataset), dtype=np.int64)
    if maximum_items is None or len(indices) <= int(maximum_items):
        return indices
    rng = np.random.default_rng(SEED + int(seed_offset))
    return rng.choice(indices, size=int(maximum_items), replace=False)


def denormalize_target(y_norm: torch.Tensor) -> torch.Tensor:
    return y_norm * target_std_t + target_mean_t


def normalize_target_np(y_phys: np.ndarray) -> np.ndarray:
    return ((y_phys.astype(np.float32) - target_mean[None, :]) / target_std[None, :]).astype(np.float32)


def normalize_input_np(x_phys_active: np.ndarray) -> np.ndarray:
    x = (x_phys_active.astype(np.float32) - input_mean[None, :]) / input_std[None, :]
    return np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)


def relative_l2(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    diff = (pred - target).reshape(pred.shape[0], -1)
    ref = target.reshape(target.shape[0], -1)
    return torch.linalg.norm(diff, dim=1) / torch.linalg.norm(ref, dim=1).clamp_min(eps)


def grouped_training_loss(pred: torch.Tensor, target: torch.Tensor):
    pred_phys = denormalize_target(pred)
    target_phys = denormalize_target(target)
    mse = torch.mean((pred_phys - target_phys) ** 2)
    vel_mse = torch.mean((pred_phys[..., :3] - target_phys[..., :3]) ** 2)
    grad_mse = torch.mean((pred_phys[..., 3:] - target_phys[..., 3:]) ** 2) if pred.shape[-1] > 3 else torch.zeros((), device=pred.device)
    return mse, vel_mse.detach(), grad_mse.detach()


def finite_mean(values):
    finite = [float(v) for v in values if np.isfinite(v)]
    return float(np.mean(finite)) if finite else np.nan


def frame_number_for_sort(frame_id: int) -> int:
    context = frame_context_as_dict(int(frame_id))
    fallback = frame_ranges[int(frame_id)][1] if int(frame_id) < len(frame_ranges) else int(frame_id)
    return int(float(context.get('frame', context.get('fr', fallback))))


def case_name_for_frame(frame_id: int) -> str:
    context = frame_context_as_dict(int(frame_id))
    fallback = frame_ranges[int(frame_id)][0] if int(frame_id) < len(frame_ranges) else 'unknown'
    return str(context.get('case', context.get('case_name', fallback)))


def time_value_for_frame(frame_id: int) -> float:
    context = frame_context_as_dict(int(frame_id))
    return float(context.get('time', frame_number_for_sort(frame_id) * float(context.get('dt', 1.0))))


In [ ]:
class ParticleUGradUAnalysisDataset(Dataset):
    def __init__(self, frame_ids: Sequence[int], split_name: str, max_input_particles: Optional[int], max_output_points: Optional[int], seed_offset: int = 0):
        self.frame_ids = np.asarray(frame_ids, dtype=np.int64)
        self.split_name = str(split_name)
        self.max_input_particles = max_input_particles
        self.max_output_points = max_output_points
        self.seed_offset = int(seed_offset)

    def __len__(self) -> int:
        return int(len(self.frame_ids))

    def __getitem__(self, index: int) -> Dict:
        frame_id = int(self.frame_ids[int(index)])
        features_all = np.asarray(inputs_by_frame[frame_id], dtype=np.float32)
        targets_all = np.asarray(targets_by_frame[frame_id], dtype=np.float32)[:, target_indices]
        n = min(features_all.shape[0], targets_all.shape[0])
        in_idx = sample_indices(n, self.max_input_particles, SEED + self.seed_offset + frame_id)
        out_idx_local = sample_indices(len(in_idx), self.max_output_points, SEED + self.seed_offset + 100000 + frame_id)
        query_idx = in_idx[out_idx_local]

        input_features_all = features_all[in_idx]
        query_features_all = features_all[query_idx]
        target_phys = targets_all[query_idx]

        context = frame_context_as_dict(frame_id)
        metadata = {
            'frame_id': frame_id,
            'split': self.split_name,
            'case': case_name_for_frame(frame_id),
            'frame': str(context.get('frame', context.get('fr', frame_number_for_sort(frame_id)))),
            'aoa_deg': float(context.get('aoa_deg', np.nan)),
            'particle_count_total': int(n),
            'particle_count_input': int(len(in_idx)),
            'particle_count_query': int(len(query_idx)),
        }
        return {
            'input_geom': torch.from_numpy(normalize_xyz(input_features_all[:, coord_feature_indices])),
            'x': torch.from_numpy(normalize_input_np(input_features_all[:, active_input_feature_indices])),
            'output_queries': torch.from_numpy(normalize_xyz(query_features_all[:, coord_feature_indices])),
            'y': torch.from_numpy(normalize_target_np(target_phys)),
            'query_raw_features': torch.from_numpy(query_features_all[:, active_input_feature_indices].astype(np.float32)),
            'query_xyz_raw': torch.from_numpy(query_features_all[:, coord_feature_indices].astype(np.float32)),
            'target_phys': torch.from_numpy(np.nan_to_num(target_phys).astype(np.float32)),
            'query_indices': torch.from_numpy(query_idx.astype(np.int64)),
            'frame_id': torch.tensor(frame_id, dtype=torch.long),
            'metadata': metadata,
        }


def collate_analysis(batch):
    assert len(batch) == 1, 'This notebook uses batch_size=1 because particle counts vary by frame.'
    item = batch[0]
    out = {}
    for key, value in item.items():
        if torch.is_tensor(value) and key not in {'frame_id'}:
            out[key] = value.unsqueeze(0)
        else:
            out[key] = value
    out['frame_id'] = item['frame_id'].view(1)
    return out


def make_loader(frame_ids: Sequence[int], split_name: str, max_input_particles: Optional[int] = None, max_output_points: Optional[int] = None, shuffle: bool = False):
    ds = ParticleUGradUAnalysisDataset(frame_ids, split_name, max_input_particles, max_output_points)
    return ds, DataLoader(ds, batch_size=1, shuffle=shuffle, num_workers=0, collate_fn=collate_analysis)

analysis_max_input_particles = int(os.environ.get('GINO_ANALYSIS_MAX_INPUT_PARTICLES', CFG.get('maximum_input_particles', 1000)))
analysis_max_output_points = int(os.environ.get('GINO_ANALYSIS_MAX_OUTPUT_POINTS', CFG.get('maximum_eval_output_points', 4096)))
plot_max_output_points = int(os.environ.get('GINO_ANALYSIS_PLOT_POINTS', min(12000, max(analysis_max_output_points, 4096))))

training_dataset, training_loader = make_loader(train_frame_ids, 'training', analysis_max_input_particles, analysis_max_output_points)
validation_dataset, validation_loader = make_loader(val_id_frame_ids, 'validation', analysis_max_input_particles, analysis_max_output_points)
validation_angle_dataset, validation_angle_loader = make_loader(val_angle_frame_ids, 'validation_angle', analysis_max_input_particles, analysis_max_output_points)
testing_dataset, testing_loader = make_loader(testing_frame_ids, 'testing_all', analysis_max_input_particles, analysis_max_output_points)
testing_normal_dataset, testing_normal_loader = make_loader(test_normal_frame_ids, 'testing_normal', analysis_max_input_particles, analysis_max_output_points)
testing_super_resolution_dataset, testing_super_resolution_loader = make_loader(test_spatial_sr_frame_ids, 'testing_super_resolution', analysis_max_input_particles, analysis_max_output_points)
testing_unseen_angle_dataset, testing_unseen_angle_loader = make_loader(test_unseen_angle_frame_ids, 'testing_unseen_angle', analysis_max_input_particles, analysis_max_output_points)

print('Analysis caps:', {'input_particles': analysis_max_input_particles, 'output_points': analysis_max_output_points, 'plot_points': plot_max_output_points})


In [ ]:
def make_latent_queries(res: int, device: torch.device) -> torch.Tensor:
    line = torch.linspace(0.0, 1.0, int(res), dtype=torch.float32, device=device)
    xx, yy, zz = torch.meshgrid(line, line, line, indexing='ij')
    return torch.stack([xx, yy, zz], dim=-1).unsqueeze(0)


LATENT_QUERIES = make_latent_queries(int(CFG.get('latent_res', 8)), DEVICE)


def build_neuralop_gino(cfg: Dict) -> nn.Module:
    if GINO is None:
        raise RuntimeError('neuralop.models.GINO is not available in this kernel.') from GINO_IMPORT_ERROR
    kwargs = dict(
        in_channels=len(feature_names),
        out_channels=len(target_names),
        gno_coord_dim=3,
        in_gno_radius=cfg.get('in_gno_radius', 0.35),
        out_gno_radius=cfg.get('out_gno_radius', 0.40),
        in_gno_transform_type=cfg.get('in_gno_transform_type', 'nonlinear_kernelonly'),
        out_gno_transform_type=cfg.get('out_gno_transform_type', 'linear'),
        gno_embed_channels=cfg.get('gno_embed_channels', 32),
        gno_use_open3d=cfg.get('gno_use_open3d', False),
        gno_use_torch_scatter=cfg.get('gno_use_torch_scatter', False),
        fno_n_modes=tuple(cfg.get('fno_n_modes', (4, 4, 4))),
        fno_hidden_channels=cfg.get('fno_hidden_channels', 32),
        fno_n_layers=cfg.get('fno_n_layers', 4),
        projection_channel_ratio=cfg.get('projection_channel_ratio', 2),
    )
    accepted = set(inspect.signature(GINO).parameters)
    return GINO(**{k: v for k, v in kwargs.items() if k in accepted}).to(DEVICE)


class PointwiseLatentGINO(nn.Module):
    # Included for compatibility with future field-reconstruction checkpoints. The current task1 checkpoint uses NeuralOperator GINO.
    def __init__(self, in_channels: int, out_channels: int, cfg: Dict):
        super().__init__()
        from neuralop.layers.gno_block import GNOBlock
        hidden = int(cfg.get('fno_hidden_channels', 32))
        self.latent_res = int(cfg.get('latent_res', 8))
        self.lift = nn.Sequential(nn.Linear(in_channels, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.in_gno = GNOBlock(
            in_channels=hidden,
            out_channels=hidden,
            coord_dim=3,
            radius=float(cfg.get('in_gno_radius', 0.35)),
            transform_type=str(cfg.get('in_gno_transform_type', 'nonlinear_kernelonly')),
            reduction='mean',
            pos_embedding_type='transformer',
            pos_embedding_channels=12,
            channel_mlp_layers=[hidden, hidden, hidden],
            use_torch_scatter_reduce=bool(cfg.get('gno_use_torch_scatter', False)),
            use_open3d_neighbor_search=bool(cfg.get('gno_use_open3d', False)),
        )
        layers = []
        for _ in range(max(int(cfg.get('fno_n_layers', 4)), 1)):
            layers += [nn.Conv3d(hidden, hidden, kernel_size=3, padding=1), nn.GELU()]
        self.latent_mixer = nn.Sequential(*layers)
        width = int(cfg.get('pointwise_hidden', 96))
        mlp_layers = []
        for i in range(max(int(cfg.get('pointwise_layers', 3)), 1)):
            mlp_layers += [nn.Linear(hidden + 3 if i == 0 else width, width), nn.GELU()]
        mlp_layers.append(nn.Linear(width, out_channels))
        self.decoder = nn.Sequential(*mlp_layers)

    def forward(self, input_geom: torch.Tensor, latent_queries: torch.Tensor, output_queries: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        outs = []
        base_latent = latent_queries[0]
        for b in range(int(x.shape[0])):
            h = self.lift(x[b])
            latent = self.in_gno(y=base_latent, x=input_geom[b], f_y=h)
            if latent.ndim == 3:
                latent = latent.squeeze(0)
            r = self.latent_res
            grid = latent.reshape(r, r, r, -1).permute(3, 0, 1, 2).unsqueeze(0)
            grid = self.latent_mixer(grid)
            q = output_queries[b].clamp(0.0, 1.0)
            sample_grid = (q * 2.0 - 1.0).view(1, -1, 1, 1, 3)
            sampled = F.grid_sample(grid, sample_grid, align_corners=True, mode='bilinear')
            sampled = sampled.squeeze(0).squeeze(-1).squeeze(-1).transpose(0, 1)
            outs.append(self.decoder(torch.cat([sampled, q], dim=-1)))
        return torch.stack(outs, dim=0)


def build_model_from_checkpoint() -> nn.Module:
    if architecture == 'neuralop_gino':
        return build_neuralop_gino(CFG)
    return PointwiseLatentGINO(len(feature_names), len(target_names), CFG).to(DEVICE)


model = build_model_from_checkpoint()
model.load_state_dict(state_dict, strict=True)
model.eval()
print('Latent queries    :', tuple(LATENT_QUERIES.shape))
print('Total parameters  :', f'{count_model_params(model):,}')
print('Trainable params  :', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


In [ ]:
def move_batch(batch: Dict) -> Dict:
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


def predict(model: nn.Module, batch: Dict) -> torch.Tensor:
    return model(
        input_geom=batch['input_geom'],
        latent_queries=LATENT_QUERIES,
        output_queries=batch['output_queries'],
        x=batch['x'],
    )


def component_metrics(prediction: torch.Tensor, target: torch.Tensor, minimum_norm: float = 1e-12) -> Dict:
    difference = prediction - target
    target_norm = torch.linalg.norm(target.reshape(-1))
    difference_norm = torch.linalg.norm(difference.reshape(-1))
    relative_error = np.nan if target_norm.item() <= minimum_norm else float((difference_norm / target_norm).item())
    return {
        'relative_error': relative_error,
        'rmse': float(torch.sqrt(torch.mean(difference ** 2)).item()),
        'mae': float(torch.mean(torch.abs(difference)).item()),
        'target_norm': float(target_norm.item()),
        'target_rms': float(torch.sqrt(torch.mean(target ** 2)).item()),
    }


def physical_metrics(prediction_normalized: torch.Tensor, target_normalized: torch.Tensor) -> Dict:
    prediction = denormalize_target(prediction_normalized)
    target = denormalize_target(target_normalized)
    full = component_metrics(prediction, target)
    velocity = component_metrics(prediction[..., :3], target[..., :3])
    gradient = component_metrics(prediction[..., 3:], target[..., 3:]) if prediction.shape[-1] > 3 else {k: np.nan for k in full}
    return {
        'relative_error': full['relative_error'],
        'rmse': full['rmse'],
        'mae': full['mae'],
        'target_norm': full['target_norm'],
        'target_rms': full['target_rms'],
        'velocity_relative_error': velocity['relative_error'],
        'velocity_rmse': velocity['rmse'],
        'velocity_mae': velocity['mae'],
        'velocity_gradient_relative_error': gradient['relative_error'],
        'velocity_gradient_rmse': gradient['rmse'],
        'velocity_gradient_mae': gradient['mae'],
    }


@torch.no_grad()
def evaluate_model(data_loader: DataLoader, maximum_frames: Optional[int] = 20) -> Dict:
    if len(data_loader.dataset) == 0:
        return {'frames_evaluated': 0, 'relative_error': np.nan, 'rmse': np.nan, 'mae': np.nan, 'velocity_relative_error': np.nan, 'velocity_rmse': np.nan, 'velocity_gradient_relative_error': np.nan, 'velocity_gradient_rmse': np.nan}
    model.eval()
    collected = []
    sampled_indices = sample_dataset_indices(data_loader.dataset, maximum_frames, seed_offset=67)
    for dataset_index in sampled_indices:
        batch = move_batch(collate_analysis([data_loader.dataset[int(dataset_index)]]))
        prediction = predict(model, batch).float()
        collected.append(physical_metrics(prediction, batch['y'].float()))
    return {
        'frames_evaluated': int(len(collected)),
        'relative_error': finite_mean([m['relative_error'] for m in collected]),
        'rmse': finite_mean([m['rmse'] for m in collected]),
        'mae': finite_mean([m['mae'] for m in collected]),
        'velocity_relative_error': finite_mean([m['velocity_relative_error'] for m in collected]),
        'velocity_rmse': finite_mean([m['velocity_rmse'] for m in collected]),
        'velocity_gradient_relative_error': finite_mean([m['velocity_gradient_relative_error'] for m in collected]),
        'velocity_gradient_rmse': finite_mean([m['velocity_gradient_rmse'] for m in collected]),
    }


In [ ]:
history = json.loads(HISTORY_PATH.read_text()) if HISTORY_PATH.exists() else checkpoint.get('history', {})
if not history or len(history.get('epoch', [])) == 0:
    print('[skip] No history data found.')
else:
    epochs = np.asarray(history['epoch'])
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), constrained_layout=True)

    axes[0].plot(epochs, history.get('train_loss', []), marker='o', label='train loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss in physical units')
    axes[0].set_title('Training loss')
    axes[0].grid(alpha=0.25)
    axes[0].legend(frameon=False)

    axes[1].plot(epochs, history.get('train_rel_l2_norm', []), marker='o', label='train norm')
    axes[1].plot(epochs, history.get('val_id_rel_l2_phys', []), marker='o', label='val id')
    axes[1].plot(epochs, history.get('val_angle_rel_l2_phys', []), marker='o', label='val angle')
    axes[1].plot(epochs, history.get('test_rel_l2_phys', []), marker='o', label='test')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Relative L2')
    axes[1].set_yscale('log')
    axes[1].set_title('Relative-error curves')
    axes[1].grid(alpha=0.25, which='both')
    axes[1].legend(frameon=False)

    axes[2].plot(epochs, history.get('lr', []), marker='o', color='tab:orange', label='learning rate')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning rate')
    axes[2].set_title('Learning-rate schedule')
    axes[2].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    axes[2].grid(alpha=0.25)
    axes[2].legend(frameon=False)

    plot_path = PLOTS_DIR / 'learning_curves_current_gino.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)

    fig, axis = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
    axis.plot(epochs, history.get('train_primary_loss', []), marker='o', label='velocity loss')
    axis.plot(epochs, history.get('train_secondary_loss', []), marker='o', label='gradU loss')
    axis.set_xlabel('Epoch')
    axis.set_ylabel('Grouped training loss')
    axis.set_yscale('log')
    axis.set_title('Velocity vs gradU loss')
    axis.grid(alpha=0.25, which='both')
    axis.legend(frameon=False)
    plot_path = PLOTS_DIR / 'grouped_training_losses.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


In [ ]:
final_metrics = {
    'training': evaluate_model(training_loader, maximum_frames=12),
    'validation': evaluate_model(validation_loader, maximum_frames=20),
    'validation_angle': evaluate_model(validation_angle_loader, maximum_frames=20),
    'testing_all': evaluate_model(testing_loader, maximum_frames=20),
    'testing_normal': evaluate_model(testing_normal_loader, maximum_frames=20),
    'testing_super_resolution': evaluate_model(testing_super_resolution_loader, maximum_frames=20),
    'testing_unseen_angle': evaluate_model(testing_unseen_angle_loader, maximum_frames=20),
}
final_metrics = {k: v for k, v in final_metrics.items() if v['frames_evaluated'] > 0}
print(json.dumps(final_metrics, indent=2))

if final_metrics:
    fig, axis = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    split_names = list(final_metrics.keys())
    values = [final_metrics[name]['relative_error'] for name in split_names]
    bars = axis.bar(split_names, values, color=plt.cm.tab10(np.arange(len(split_names)) % 10))
    axis.tick_params(axis='x', rotation=25)
    axis.set_ylabel('Relative L2 in physical units')
    axis.set_title('Final checkpoint error by split')
    axis.grid(axis='y', alpha=0.25)
    finite_values = [v for v in values if np.isfinite(v)]
    if finite_values and max(finite_values) / max(min(finite_values), 1e-12) > 100:
        axis.set_yscale('log')
    for bar, value in zip(bars, values):
        if np.isfinite(value):
            axis.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.3g}', ha='center', va='bottom', fontsize=9)
    plot_path = PLOTS_DIR / 'final_relative_error_by_split_current_gino.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


In [ ]:
@torch.no_grad()
def collect_prediction_samples(dataset, maximum_frames=6, maximum_points_per_frame=4096):
    if len(dataset) == 0:
        return None, None
    true_chunks = []
    predicted_chunks = []
    for dataset_index in sample_dataset_indices(dataset, maximum_frames, seed_offset=79):
        frame_id = int(dataset.frame_ids[int(dataset_index)])
        temp_ds = ParticleUGradUAnalysisDataset([frame_id], dataset.split_name, analysis_max_input_particles, maximum_points_per_frame, seed_offset=500)
        batch = move_batch(collate_analysis([temp_ds[0]]))
        prediction_normalized = predict(model, batch).float()
        true_physical = denormalize_target(batch['y']).squeeze(0).cpu().numpy()
        predicted_physical = denormalize_target(prediction_normalized).squeeze(0).cpu().numpy()
        true_chunks.append(true_physical)
        predicted_chunks.append(predicted_physical)
    return np.concatenate(true_chunks, axis=0), np.concatenate(predicted_chunks, axis=0)


def plot_scatter_for_split(split_name, dataset):
    true_values, predicted_values = collect_prediction_samples(dataset)
    if true_values is None:
        print(f'[{split_name}] no data available; skipping scatter plot.')
        return
    true_velocity = np.linalg.norm(true_values[:, :3], axis=1)
    predicted_velocity = np.linalg.norm(predicted_values[:, :3], axis=1)
    true_gradient = np.linalg.norm(true_values[:, 3:], axis=1) if true_values.shape[1] > 3 else np.zeros(len(true_values))
    predicted_gradient = np.linalg.norm(predicted_values[:, 3:], axis=1) if predicted_values.shape[1] > 3 else np.zeros(len(predicted_values))

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), constrained_layout=True)
    for axis, true_array, predicted_array, title in [
        (axes[0], true_velocity, predicted_velocity, '|u|'),
        (axes[1], true_gradient, predicted_gradient, '|gradU|'),
    ]:
        if true_array.shape[0] > 30000:
            picked = np.random.default_rng(SEED).choice(true_array.shape[0], size=30000, replace=False)
            true_array = true_array[picked]
            predicted_array = predicted_array[picked]
        both = np.concatenate([true_array, predicted_array])
        low = float(np.nanquantile(both, 0.01))
        high = float(np.nanquantile(both, 0.99))
        if not np.isfinite(low) or not np.isfinite(high) or low >= high:
            low, high = float(np.nanmin(both)), float(np.nanmax(both) + 1e-8)
        axis.scatter(true_array, predicted_array, s=3, alpha=0.25)
        axis.plot([low, high], [low, high], color='black', linewidth=1.2, linestyle='--')
        axis.set_xlabel(f'true {title}')
        axis.set_ylabel(f'predicted {title}')
        axis.set_title(f'{split_name}: {title}')
        axis.grid(alpha=0.25)
    plot_path = PLOTS_DIR / f'{split_name.lower()}_prediction_scatter_current_gino.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


for split_name, split_dataset in [
    ('training', training_dataset),
    ('validation', validation_dataset),
    ('testing_all', testing_dataset),
    ('testing_normal', testing_normal_dataset),
    ('testing_super_resolution', testing_super_resolution_dataset),
    ('testing_unseen_angle', testing_unseen_angle_dataset),
]:
    plot_scatter_for_split(split_name, split_dataset)


In [ ]:
@torch.no_grad()
def predict_one_frame(dataset, dataset_index=0, maximum_particles_for_plot=12000):
    if len(dataset) == 0:
        raise RuntimeError('Cannot predict from an empty dataset.')
    dataset_index = min(max(int(dataset_index), 0), len(dataset) - 1)
    frame_id = int(dataset.frame_ids[dataset_index])
    temp_ds = ParticleUGradUAnalysisDataset([frame_id], dataset.split_name, analysis_max_input_particles, maximum_particles_for_plot, seed_offset=900)
    batch = move_batch(collate_analysis([temp_ds[0]]))
    prediction_normalized = predict(model, batch).float()
    target_physical = denormalize_target(batch['y']).squeeze(0).cpu().numpy()
    prediction_physical = denormalize_target(prediction_normalized).squeeze(0).cpu().numpy()
    coordinates = batch['query_xyz_raw'].squeeze(0).cpu().numpy()
    metadata = batch['metadata']
    return coordinates, target_physical, prediction_physical, metadata


def robust_color_limits(values, lower=0.02, upper=0.98):
    finite_values = np.asarray(values)[np.isfinite(values)]
    if finite_values.size == 0:
        return 0.0, 1.0
    low = float(np.quantile(finite_values, lower))
    high = float(np.quantile(finite_values, upper))
    if low >= high:
        high = low + 1e-6
    return low, high


def plot_prediction_example(split_name, dataset, dataset_index=0):
    if len(dataset) == 0:
        print(f'[{split_name}] no data available; skipping prediction image.')
        return
    coordinates, true_values, predicted_values, metadata = predict_one_frame(dataset, dataset_index=dataset_index, maximum_particles_for_plot=plot_max_output_points)
    true_velocity = np.linalg.norm(true_values[:, :3], axis=1)
    predicted_velocity = np.linalg.norm(predicted_values[:, :3], axis=1)
    velocity_error = predicted_velocity - true_velocity
    true_gradient = np.linalg.norm(true_values[:, 3:], axis=1) if true_values.shape[1] > 3 else np.zeros(len(true_values))
    predicted_gradient = np.linalg.norm(predicted_values[:, 3:], axis=1) if predicted_values.shape[1] > 3 else np.zeros(len(predicted_values))
    gradient_error = predicted_gradient - true_gradient

    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)
    velocity_low, velocity_high = robust_color_limits(np.concatenate([true_velocity, predicted_velocity]))
    gradient_low, gradient_high = robust_color_limits(np.concatenate([true_gradient, predicted_gradient]))
    velocity_error_limit = max(abs(robust_color_limits(velocity_error)[0]), abs(robust_color_limits(velocity_error)[1]), 1e-12)
    gradient_error_limit = max(abs(robust_color_limits(gradient_error)[0]), abs(robust_color_limits(gradient_error)[1]), 1e-12)

    plot_items = [
        (0, 0, true_velocity, '|u| true', 'viridis', velocity_low, velocity_high),
        (0, 1, predicted_velocity, '|u| prediction', 'viridis', velocity_low, velocity_high),
        (0, 2, velocity_error, '|u| signed error', 'RdBu_r', -velocity_error_limit, velocity_error_limit),
        (1, 0, true_gradient, '|gradU| true', 'magma', gradient_low, gradient_high),
        (1, 1, predicted_gradient, '|gradU| prediction', 'magma', gradient_low, gradient_high),
        (1, 2, gradient_error, '|gradU| signed error', 'RdBu_r', -gradient_error_limit, gradient_error_limit),
    ]
    for row, column, values, title, color_map, low, high in plot_items:
        scatter = axes[row, column].scatter(coordinates[:, 0], coordinates[:, 2], c=values, s=3, cmap=color_map, vmin=low, vmax=high)
        axes[row, column].set_title(title)
        axes[row, column].set_xlabel('x')
        axes[row, column].set_ylabel('z')
        axes[row, column].grid(alpha=0.18)
        plt.colorbar(scatter, ax=axes[row, column], fraction=0.046, pad=0.02)
    fig.suptitle(f"{split_name}: case {metadata['case']}, frame {metadata['frame']}, particles shown {coordinates.shape[0]}", fontsize=13)
    plot_path = PLOTS_DIR / f'{split_name.lower()}_prediction_example_current_gino.png'
    fig.savefig(plot_path, dpi=240, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


for split_name, split_dataset in [
    ('training', training_dataset),
    ('validation', validation_dataset),
    ('testing_all', testing_dataset),
    ('testing_normal', testing_normal_dataset),
    ('testing_super_resolution', testing_super_resolution_dataset),
    ('testing_unseen_angle', testing_unseen_angle_dataset),
]:
    index = min(180, max(len(split_dataset) - 1, 0))
    plot_prediction_example(split_name, split_dataset, dataset_index=index)


## Temporal Traces

These cells track one particle index through a train case and a held-out case. They use the raw frame arrays directly, so they are independent of the random query subsampling used for model evaluation.


In [ ]:
def frame_ids_for_case(frame_ids, case_name):
    selected = [int(frame_id) for frame_id in np.asarray(frame_ids, dtype=np.int64) if case_name_for_frame(int(frame_id)) == str(case_name)]
    return sorted(selected, key=frame_number_for_sort)


def first_case_from_frames(frame_ids):
    frame_ids = np.asarray(frame_ids, dtype=np.int64)
    if len(frame_ids) == 0:
        raise RuntimeError('Cannot choose a case from an empty frame list.')
    first_frame_id = int(sorted(frame_ids, key=frame_number_for_sort)[0])
    return case_name_for_frame(first_frame_id)


def extract_temporal_target_series(frame_ids, particle_index=0, normalized=True):
    time_values, velocity_magnitudes, gradient_magnitudes, used_frames = [], [], [], []
    for frame_id in frame_ids:
        target_frame = targets_by_frame_raw_active[int(frame_id)]
        if particle_index >= target_frame.shape[0]:
            continue
        particle_target = target_frame[particle_index]
        if normalized:
            particle_target = (particle_target - target_mean) / target_std
        time_values.append(time_value_for_frame(frame_id))
        velocity_magnitudes.append(np.linalg.norm(particle_target[:3]))
        gradient_magnitudes.append(np.linalg.norm(particle_target[3:]))
        used_frames.append(frame_number_for_sort(frame_id))
    if not time_values:
        raise RuntimeError(f'particle_index={particle_index} was not available in any selected frame.')
    return {'time': np.asarray(time_values), 'velocity': np.asarray(velocity_magnitudes), 'gradient': np.asarray(gradient_magnitudes), 'frame': np.asarray(used_frames)}


particle_index = 0
available_training_cases = sorted({case_name_for_frame(int(frame_id)) for frame_id in train_frame_ids})
if len(available_training_cases) == 0:
    raise RuntimeError('No training cases available for temporal plotting.')
train_case_name = available_training_cases[0]
test_frames_source = test_unseen_angle_frame_ids if len(test_unseen_angle_frame_ids) else testing_frame_ids
test_source_name = 'testing_unseen_angle' if len(test_unseen_angle_frame_ids) else 'testing_all'
test_case_name = first_case_from_frames(test_frames_source)

train_case_frame_ids = frame_ids_for_case(train_frame_ids, train_case_name)
test_case_frame_ids = frame_ids_for_case(test_frames_source, test_case_name)
train_series = extract_temporal_target_series(train_case_frame_ids, particle_index, normalized=True)
test_series = extract_temporal_target_series(test_case_frame_ids, particle_index, normalized=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
items = [
    (axes[0, 0], train_series['time'], train_series['velocity'], f'Train normalized |u|\ncase={train_case_name}, particle={particle_index}', '#4c78a8'),
    (axes[0, 1], train_series['time'], train_series['gradient'], f'Train normalized |gradU|\ncase={train_case_name}, particle={particle_index}', '#4c78a8'),
    (axes[1, 0], test_series['time'], test_series['velocity'], f'Test normalized |u|\ncase={test_case_name}, particle={particle_index}', '#e15759'),
    (axes[1, 1], test_series['time'], test_series['gradient'], f'Test normalized |gradU|\ncase={test_case_name}, particle={particle_index}', '#e15759'),
]
for axis, x, y, title, color in items:
    axis.plot(x, y, linewidth=2.0, color=color)
    axis.set_title(title)
    axis.set_xlabel('time')
    axis.grid(alpha=0.3)
plot_path = PLOTS_DIR / f'temporal_particle_{particle_index}_normalized_u_gradU_current_gino.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


In [ ]:
use_normalized_inputs_for_temporal_plot = False
input_value_label = 'normalized input' if use_normalized_inputs_for_temporal_plot else 'raw input'


def input_feature_index(name):
    if name not in feature_names:
        raise RuntimeError(f'Missing required input feature {name!r}. Available features: {feature_names}')
    return feature_names.index(name)


def extract_temporal_input_series(frame_ids, particle_index=0):
    gamma_indices = [input_feature_index(name) for name in ['Gamma_x', 'Gamma_y', 'Gamma_z']]
    sigma_index = input_feature_index('sigma')
    time_values, position_values, gamma_values, sigma_values = [], [], [], []
    for frame_id in frame_ids:
        full_input_frame = np.asarray(inputs_by_frame[int(frame_id)], dtype=np.float32)
        active_input_frame = inputs_by_frame_raw_active[int(frame_id)]
        if particle_index >= active_input_frame.shape[0]:
            continue
        particle_active_input = active_input_frame[particle_index]
        particle_position = full_input_frame[particle_index, coord_feature_indices]
        if use_normalized_inputs_for_temporal_plot:
            particle_active_input = (particle_active_input - input_mean) / input_std
            particle_position = normalize_xyz(particle_position.reshape(1, 3))[0]
        time_values.append(time_value_for_frame(frame_id))
        position_values.append(particle_position)
        gamma_values.append(particle_active_input[gamma_indices])
        sigma_values.append(particle_active_input[sigma_index])
    if not time_values:
        raise RuntimeError(f'particle_index={particle_index} was not available in any selected frame.')
    return {'time': np.asarray(time_values), 'position': np.asarray(position_values), 'gamma': np.asarray(gamma_values), 'sigma': np.asarray(sigma_values)}


train_input_series = extract_temporal_input_series(train_case_frame_ids, particle_index)
test_input_series = extract_temporal_input_series(test_case_frame_ids, particle_index)

fig, axes = plt.subplots(3, 2, figsize=(15, 11), constrained_layout=True)
for component_id, label in enumerate(['x', 'y', 'z']):
    axes[0, 0].plot(train_input_series['time'], train_input_series['position'][:, component_id], linewidth=1.8, label=label)
    axes[0, 1].plot(test_input_series['time'], test_input_series['position'][:, component_id], linewidth=1.8, label=label)
for component_id, label in enumerate(['Gamma_x', 'Gamma_y', 'Gamma_z']):
    axes[1, 0].plot(train_input_series['time'], train_input_series['gamma'][:, component_id], linewidth=1.8, label=label)
    axes[1, 1].plot(test_input_series['time'], test_input_series['gamma'][:, component_id], linewidth=1.8, label=label)
axes[2, 0].plot(train_input_series['time'], train_input_series['sigma'], linewidth=2.0, color='#4c78a8')
axes[2, 1].plot(test_input_series['time'], test_input_series['sigma'], linewidth=2.0, color='#e15759')

titles = [
    f'Train position\ncase={train_case_name}', f'Test position\ncase={test_case_name}',
    f'Train Gamma\ncase={train_case_name}', f'Test Gamma\ncase={test_case_name}',
    f'Train sigma\ncase={train_case_name}', f'Test sigma\ncase={test_case_name}',
]
for axis, title in zip(axes.reshape(-1), titles):
    axis.set_title(title)
    axis.set_xlabel('time')
    axis.set_ylabel(input_value_label)
    axis.grid(alpha=0.3)
    if len(axis.get_lines()) > 1:
        axis.legend(frameon=False)
plot_path = PLOTS_DIR / f'temporal_particle_{particle_index}_input_features_current_gino.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


In [ ]:
@torch.no_grad()
def collect_particle_errors(dataset, maximum_frames=8, maximum_points_per_frame=4096):
    if len(dataset) == 0:
        return np.asarray([]), np.asarray([])
    velocity_errors, gradient_errors = [], []
    for dataset_index in sample_dataset_indices(dataset, maximum_frames, seed_offset=91):
        frame_id = int(dataset.frame_ids[int(dataset_index)])
        temp_ds = ParticleUGradUAnalysisDataset([frame_id], dataset.split_name, analysis_max_input_particles, maximum_points_per_frame, seed_offset=1200)
        batch = move_batch(collate_analysis([temp_ds[0]]))
        prediction = denormalize_target(predict(model, batch).float()).squeeze(0).cpu().numpy()
        target = denormalize_target(batch['y']).squeeze(0).cpu().numpy()
        velocity_true = np.linalg.norm(target[:, :3], axis=1)
        velocity_pred = np.linalg.norm(prediction[:, :3], axis=1)
        gradient_true = np.linalg.norm(target[:, 3:], axis=1) if target.shape[1] > 3 else np.zeros(len(target))
        gradient_pred = np.linalg.norm(prediction[:, 3:], axis=1) if prediction.shape[1] > 3 else np.zeros(len(prediction))
        velocity_errors.append(np.abs(velocity_pred - velocity_true) / np.maximum(velocity_true, 1e-8))
        gradient_errors.append(np.abs(gradient_pred - gradient_true) / np.maximum(gradient_true, 1e-8))
    return np.concatenate(velocity_errors), np.concatenate(gradient_errors)


def plot_error_histograms(split_name, dataset):
    velocity_errors, gradient_errors = collect_particle_errors(dataset)
    if velocity_errors.size == 0:
        print(f'[{split_name}] no data available; skipping error histogram.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
    axes[0].hist(velocity_errors[np.isfinite(velocity_errors)], bins=80, density=True, alpha=0.8)
    axes[0].set_title(f'{split_name}: velocity relative error')
    axes[0].set_xlabel('relative error')
    axes[0].set_ylabel('PDF')
    axes[0].set_yscale('log')
    axes[1].hist(gradient_errors[np.isfinite(gradient_errors)], bins=80, density=True, alpha=0.8)
    axes[1].set_title(f'{split_name}: gradU relative error')
    axes[1].set_xlabel('relative error')
    axes[1].set_ylabel('PDF')
    axes[1].set_yscale('log')
    for axis in axes:
        axis.grid(alpha=0.25)
    plot_path = PLOTS_DIR / f'{split_name.lower()}_particle_error_histograms.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


for split_name, split_dataset in [('validation', validation_dataset), ('testing_all', testing_dataset), ('testing_unseen_angle', testing_unseen_angle_dataset)]:
    plot_error_histograms(split_name, split_dataset)


In [ ]:
@torch.no_grad()
def error_vs_aoa(dataset, maximum_frames=2000):
    aoa_values, relative_errors = [], []
    for dataset_index in sample_dataset_indices(dataset, maximum_frames, seed_offset=141):
        batch = move_batch(collate_analysis([dataset[int(dataset_index)]]))
        prediction = predict(model, batch).float()
        metrics = physical_metrics(prediction, batch['y'].float())
        aoa_values.append(float(batch['metadata'].get('aoa_deg', np.nan)))
        relative_errors.append(metrics['relative_error'])
    return np.asarray(aoa_values), np.asarray(relative_errors)


for split_name, split_dataset in [('validation_angle', validation_angle_dataset), ('testing_all', testing_dataset), ('testing_unseen_angle', testing_unseen_angle_dataset)]:
    if len(split_dataset) == 0:
        continue
    aoa, error = error_vs_aoa(split_dataset)
    finite = np.isfinite(aoa) & np.isfinite(error)
    if not np.any(finite):
        print(f'[{split_name}] no finite AoA/error pairs; skipping.')
        continue
    fig, axis = plt.subplots(figsize=(6.4, 4.2), constrained_layout=True)
    axis.scatter(aoa[finite], error[finite], s=36, alpha=0.8)
    axis.set_xlabel('Angle of attack (deg)')
    axis.set_ylabel('Relative L2 in physical units')
    axis.set_title(f'Error vs angle of attack: {split_name}')
    axis.grid(alpha=0.25)
    plot_path = PLOTS_DIR / f'{split_name.lower()}_error_vs_aoa.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


In [ ]:
@torch.no_grad()
def plot_error_vs_distance(split_name, dataset, dataset_index=0):
    if len(dataset) == 0:
        print(f'[{split_name}] no data available; skipping distance plot.')
        return
    if 'geom_dist' not in feature_names:
        print('[skip] geom_dist is not one of the checkpoint input features.')
        return
    dataset_index = min(max(int(dataset_index), 0), len(dataset) - 1)
    batch = move_batch(collate_analysis([dataset[dataset_index]]))
    prediction = denormalize_target(predict(model, batch).float()).squeeze(0).cpu().numpy()
    target = denormalize_target(batch['y']).squeeze(0).cpu().numpy()
    raw_input = batch['query_raw_features'].squeeze(0).cpu().numpy()
    distance = raw_input[:, feature_names.index('geom_dist')]
    velocity_true = np.linalg.norm(target[:, :3], axis=1)
    velocity_pred = np.linalg.norm(prediction[:, :3], axis=1)
    relative_error = np.abs(velocity_pred - velocity_true) / np.maximum(velocity_true, 1e-8)
    fig, axis = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
    axis.scatter(distance, relative_error, s=3, alpha=0.3)
    axis.set_xlabel('distance from geometry')
    axis.set_ylabel('velocity magnitude relative error')
    axis.set_title(f'Error vs distance: {split_name}')
    axis.grid(alpha=0.25)
    plot_path = PLOTS_DIR / f'{split_name.lower()}_error_vs_distance.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


plot_error_vs_distance('testing_all', testing_dataset, dataset_index=0)
plot_error_vs_distance('testing_unseen_angle', testing_unseen_angle_dataset, dataset_index=0)


In [ ]:
try:
    from scipy.interpolate import griddata
    SCIPY_AVAILABLE = True
except Exception as error:
    SCIPY_AVAILABLE = False
    SCIPY_IMPORT_ERROR = error


@torch.no_grad()
def energy_spectrum_plot(split_name, dataset, dataset_index=0, grid_resolution=128):
    if not SCIPY_AVAILABLE:
        print('[skip] scipy is not available; cannot run griddata-based spectrum plot:', SCIPY_IMPORT_ERROR)
        return
    if len(dataset) == 0:
        print(f'[{split_name}] no data available; skipping spectrum plot.')
        return
    dataset_index = min(max(int(dataset_index), 0), len(dataset) - 1)
    coordinates, target, prediction, metadata = predict_one_frame(dataset, dataset_index=dataset_index, maximum_particles_for_plot=plot_max_output_points)
    # Use the x-z plane because this particle task is usually inspected in that projection.
    points = coordinates[:, [0, 2]]
    velocity_true = np.linalg.norm(target[:, :3], axis=1)
    velocity_pred = np.linalg.norm(prediction[:, :3], axis=1)
    x = points[:, 0]
    z = points[:, 1]
    grid_x, grid_z = np.meshgrid(np.linspace(x.min(), x.max(), grid_resolution), np.linspace(z.min(), z.max(), grid_resolution))
    true_grid = griddata(points, velocity_true, (grid_x, grid_z), method='linear', fill_value=0.0)
    pred_grid = griddata(points, velocity_pred, (grid_x, grid_z), method='linear', fill_value=0.0)
    true_fft = np.abs(np.fft.fft2(true_grid)) ** 2
    pred_fft = np.abs(np.fft.fft2(pred_grid)) ** 2
    true_spectrum = np.mean(np.fft.fftshift(true_fft), axis=0)
    pred_spectrum = np.mean(np.fft.fftshift(pred_fft), axis=0)
    k = np.arange(len(true_spectrum))
    fig, axis = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
    axis.loglog(k[1:], true_spectrum[1:], label='true')
    axis.loglog(k[1:], pred_spectrum[1:], label='prediction')
    axis.set_xlabel('wavenumber index k')
    axis.set_ylabel('velocity-magnitude spectrum')
    axis.set_title(f'Energy spectrum: {split_name}')
    axis.legend(frameon=False)
    axis.grid(alpha=0.25, which='both')
    plot_path = PLOTS_DIR / f'{split_name.lower()}_energy_spectrum.png'
    fig.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


energy_spectrum_plot('testing_all', testing_dataset, dataset_index=0)
